# ESG Financial Distress Analysis — Colab Pipeline (v2, 28 Apr 2026)

**Author:** Assoc. Prof. Dr. Wirapong Chansanam (with Robster assistant)

End-to-end reproducible pipeline for the manuscript
*"ESG and Financial Distress: Evidence from SET-listed firms"*.

## Changes from v1
- Altman Z″-Score now uses **+3.25 intercept** by default (Altman et al., 2017).
- Machine-learning step has an explicit **`NO_LEAKAGE`** mode that drops financial-ratio features from the feature set when the target is derived from those ratios.
- Logistic regression uses **internal ESG dimensions** (ESG/ES/SS/GS) instead of S&P scores to avoid singular-matrix errors.

## Pipeline outline
| Step | Section |
|------|---------|
| 0 | Setup, install dependencies, mount Google Drive |
| 1 | Load CSVs, build Altman Z″-Score, descriptives, OLS/Logit, ML |
| 2 | VIF, PanelOLS (FE) + First-Difference, SHAP, interaction term, Merton DD |
| 3 | Hausman specification test (rank-truncated) |

Outputs are written to `WORK_DIR` and downloaded via the Files panel.


## 0. Setup

Run this cell once. It installs missing libraries and configures input/output paths.

In [ ]:
# Install dependencies (Colab usually already has the rest)
!pip install -q linearmodels shap python-docx lxml openpyxl


In [ ]:
import os, sys, io, warnings, re
warnings.filterwarnings('ignore')

# ---------------- Path configuration ----------------
USE_DRIVE = False   # set True to read/write from Google Drive
DRIVE_DIR = "/content/drive/MyDrive/Financial Innovation"
LOCAL_DIR = "/content/work"

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    WORK_DIR = DRIVE_DIR
else:
    WORK_DIR = LOCAL_DIR

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Working directory:', WORK_DIR)
print('Files present:', os.listdir(WORK_DIR) if os.path.isdir(WORK_DIR) else '(empty)')


In [ ]:
# ---------------- Optional: upload local files ----------------
# Run this cell only if USE_DRIVE = False and you need to upload data.
# from google.colab import files
# uploaded = files.upload()
# for fname in uploaded:
#     dest = os.path.join(WORK_DIR, fname)
#     with open(dest, 'wb') as f:
#         f.write(uploaded[fname])
#     print('Saved', dest)


In [ ]:
# Common imports used throughout
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from io import StringIO
from copy import deepcopy
from scipy import stats as spstats

def read_csv(name):
    path = os.path.join(WORK_DIR, name) if not os.path.isabs(name) else name
    try:
        return pd.read_csv(path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin1')

print('Imports OK')


## 1. Main pipeline (Z″-Score, descriptives, OLS/Logit, ML)

Inputs expected in `WORK_DIR`:
- `Thailand_ESG__data_30102025.csv`
- `ALL_SET_952_ESG_2013-2025_1.csv`
- `sector_mapping.csv`

In [ ]:
# ============================================================
# STEP 1: DATA LOADING & MERGING
# ============================================================
df_base = read_csv('Thailand_ESG__data_30102025.csv')
print(f"[Base panel] {df_base.shape} firms={df_base['firm_id'].nunique()}")

df_esg = read_csv('ALL_SET_952_ESG_2013-2025_1.csv')
df_esg['ticker_clean']  = df_esg['SP_TICKER'].str.replace(r'\.BK$', '', regex=True).str.strip().str.upper()
df_base['ticker_clean'] = df_base['firm_id'].str.strip().str.upper()

df_merged = pd.merge(
    df_base,
    df_esg[['ticker_clean', 'YEAR', 'SPG_ESG_SCORE',
            'SPG_ESG_ENVIRONMENTAL_SCORE', 'SPG_ESG_SOCIAL_SCORE',
            'SPG_ESG_GOVERNANCE_ECONOMIC_SCORE']],
    left_on=['ticker_clean','year'], right_on=['ticker_clean','YEAR'], how='left'
).drop(columns=['YEAR'], errors='ignore')

df_sector = read_csv('sector_mapping.csv').rename(columns={'Symbol':'firm_id'})
df_merged = pd.merge(df_merged, df_sector, on='firm_id', how='left')

numeric_cols = ['EBIT','Total Current Liabilities','CapitalEmployed','NetIncome',
                'Total Assets','SIZE','Total Liabilities','LEVERAGE','ROCE',
                'Close','High','Low','Daily Avg. Value (M.Baht)','ROE','ROA',
                'EBIT Margin','D/E Ratio','ESG','ES','SS','GS','ESG.1',
                'SPG_ESG_SCORE','SPG_ESG_ENVIRONMENTAL_SCORE',
                'SPG_ESG_SOCIAL_SCORE','SPG_ESG_GOVERNANCE_ECONOMIC_SCORE']
for c in numeric_cols:
    if c in df_merged.columns:
        df_merged[c] = pd.to_numeric(df_merged[c], errors='coerce')

print('Merged:', df_merged.shape, 'years:', sorted(df_merged['year'].dropna().unique()))


In [ ]:
# ============================================================
# STEP 2: ALTMAN Z''-Score (Emerging Markets / Non-manufacturing)
# Z'' = 3.25 + 6.56 X1 + 3.26 X2 + 6.72 X3 + 1.05 X4
# (Altman et al., 2017 - Journal of International Financial
#  Management & Accounting 28(2):131-171)
# X1 = Working Capital / TA       (here proxied by (CapEmp - TL)/TA)
# X2 = Retained Earnings / TA     (here proxied by NetIncome / TA)
# X3 = EBIT / TA
# X4 = Book Value of Equity / TL
# ============================================================
df = df_merged.copy()
eps = 1e-9
df['X1'] = (df['CapitalEmployed'] - df['Total Liabilities']) / (df['Total Assets'] + eps)
df['X2'] = df['NetIncome'] / (df['Total Assets'] + eps)
df['X3'] = df['EBIT'] / (df['Total Assets'] + eps)
df['X4'] = (df['Total Assets'] - df['Total Liabilities']) / (df['Total Liabilities'].abs() + eps)

INCLUDE_CONSTANT = True       # set False only to reproduce the legacy no-intercept variant
const = 3.25 if INCLUDE_CONSTANT else 0.0
df['Z_score'] = const + 6.56*df['X1'] + 3.26*df['X2'] + 6.72*df['X3'] + 1.05*df['X4']

def classify_z(z):
    if pd.isna(z): return np.nan
    if z > 2.6:    return 'Safe Zone'
    if z > 1.1:    return 'Grey Zone'
    return 'Distress Zone'

df['distress_zone']     = df['Z_score'].apply(classify_z)
df['financial_distress'] = (df['Z_score'] <= 1.1).astype(int)

print(df['Z_score'].describe().round(4).to_string())
print('\nZones:')
print(df['distress_zone'].value_counts().to_string())


In [ ]:
# ============================================================
# STEP 3-4: Descriptives + correlation heatmap
# ============================================================
print('Z by year:')
print(df.groupby('year')['Z_score'].agg(['mean','median','std','min','max']).round(3).to_string())

corr_cols = [c for c in
             ['Z_score','financial_distress',
              'SPG_ESG_SCORE','SPG_ESG_ENVIRONMENTAL_SCORE',
              'SPG_ESG_SOCIAL_SCORE','SPG_ESG_GOVERNANCE_ECONOMIC_SCORE',
              'ESG','ES','SS','GS','SIZE','LEVERAGE','ROA','ROE','D/E Ratio','EBIT Margin']
             if c in df.columns]
corr = df[corr_cols].corr(method='pearson')
fig,ax = plt.subplots(figsize=(14,11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size':7})
ax.set_title('Pearson Correlation Heatmap - ESG & Financial Variables', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Saved correlation_heatmap.png')


In [ ]:
# ============================================================
# STEP 5: Pooled OLS + Logit (year FE)
# We use the internal ESG/ES/SS/GS columns by default to avoid
# singular-matrix issues from sparsely-matched S&P scores.
# ============================================================
USE_INTERNAL_ESG = True
if USE_INTERNAL_ESG or df['SPG_ESG_SCORE'].notna().sum() <= 20:
    esg_iv,e_iv,s_iv,g_iv = 'ESG','ES','SS','GS'
    print('Using internal ESG columns (preferred)')
else:
    esg_iv,e_iv,s_iv,g_iv = ('SPG_ESG_SCORE','SPG_ESG_ENVIRONMENTAL_SCORE',
                              'SPG_ESG_SOCIAL_SCORE','SPG_ESG_GOVERNANCE_ECONOMIC_SCORE')
    print('Using S&P ESG scores')

iv_cols = [c for c in [esg_iv,e_iv,s_iv,g_iv,'SIZE','LEVERAGE','ROA'] if c in df.columns]
year_d = pd.get_dummies(df['year'].astype(str), prefix='yr', drop_first=True)
reg_df = pd.concat([df[['Z_score','financial_distress']+iv_cols].copy(), year_d], axis=1).dropna()
print(f'  Reg N={len(reg_df)} cols={iv_cols}')
X = sm.add_constant(reg_df[iv_cols + list(year_d.columns)].astype(float))

print('\n--- OLS (Z_score) ---')
ols_m = sm.OLS(reg_df['Z_score'].astype(float), X).fit(cov_type='HC1')
print(ols_m.summary2().tables[1].to_string())

print('\n--- Logit (financial_distress) ---')
try:
    lg = sm.Logit(reg_df['financial_distress'].astype(float), X).fit(maxiter=500, disp=False)
    print(lg.summary2().tables[1].to_string())
    print(f'Pseudo-R\u00b2 = {lg.prsquared:.4f}')
except Exception as e:
    print('Logit failed:', e)


In [ ]:
# ============================================================
# STEP 6: Random Forest + Gradient Boosting
# Two modes:
#   NO_LEAKAGE = True  -> ESG-only features (avoids leakage from
#                         financial ratios that already define Z-score)
#   NO_LEAKAGE = False -> include financial controls (results may
#                         look excellent but are partly spurious)
# ============================================================
NO_LEAKAGE = True

if NO_LEAKAGE:
    ml_features = [c for c in ['ESG','ES','SS','GS'] if c in df.columns]
    print('ML mode: NO-LEAKAGE (ESG-only features)')
else:
    ml_features = [c for c in [esg_iv,e_iv,s_iv,g_iv,'SIZE','LEVERAGE','ROA','ROE','D/E Ratio','EBIT Margin']
                   if c in df.columns]
    print('ML mode: legacy (with financial controls; LEAKAGE-aware)')

ml_df = df[ml_features+['financial_distress']].dropna()
print('ML dataset:', ml_df.shape, 'class:', ml_df['financial_distress'].value_counts().to_dict())

if len(ml_df) >= 30 and ml_df['financial_distress'].nunique() == 2:
    X_ml,y_ml = ml_df[ml_features].values, ml_df['financial_distress'].values
    Xtr,Xte,ytr,yte = train_test_split(X_ml,y_ml,test_size=0.2,stratify=y_ml,random_state=42)
    importance = {}
    for name,clf in [
        ('Random Forest',     RandomForestClassifier(n_estimators=200,max_depth=6,random_state=42,n_jobs=-1)),
        ('Gradient Boosting', GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,random_state=42)),
    ]:
        clf.fit(Xtr,ytr)
        yp = clf.predict(Xte); pp = clf.predict_proba(Xte)[:,1]
        try: auc = roc_auc_score(yte,pp)
        except: auc = float('nan')
        print(f'{name}: acc={accuracy_score(yte,yp):.4f} auc={auc:.4f}')
        importance[name] = clf.feature_importances_

    fig,axes = plt.subplots(1,2,figsize=(16,6))
    for ax,(name,imps) in zip(axes,importance.items()):
        idx = np.argsort(imps)[::-1]
        ax.barh([ml_features[i] for i in idx][::-1], imps[idx][::-1], color='steelblue')
        ax.set_title(name); ax.set_xlabel('Importance')
    plt.suptitle('Feature Importance - Target: Financial Distress', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR,'feature_importance.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print('Saved feature_importance.png')

df.to_csv(os.path.join(WORK_DIR,'merged_esg_distress_panel.csv'), index=False, encoding='utf-8-sig')
print('Saved merged_esg_distress_panel.csv', df.shape)


## 2. Pipeline v2 — VIF / PanelOLS / SHAP / Interaction / Merton DD

In [ ]:
# ============================================================
# STEP 2.1: VIF analysis
# ============================================================
def compute_vif(sub, feats, name):
    Xc = sm.add_constant(sub[feats].dropna())
    rows = []
    for i,col in enumerate(Xc.columns):
        if col == 'const': continue
        v = variance_inflation_factor(Xc.values.astype(float), i)
        flag = 'HIGH' if v > 10 else ('MODERATE' if v >= 5 else 'OK')
        rows.append({'Model':name,'Feature':col,'VIF':round(v,4),'Status':flag})
    return pd.DataFrame(rows)

specs = {
    'M1_Baseline':         ['LEVERAGE','SIZE','ROA'],
    'M2_ESG_Main':         ['LEVERAGE','SIZE','ROA','ESG','ES','SS','GS'],
    'M3_GScore_Only':      ['LEVERAGE','SIZE','ROA','GS'],
    'M4_Interaction_Prep': ['LEVERAGE','SIZE','ROA','GS'],
}
vif_rows = []
for name,feats in specs.items():
    af = [f for f in feats if f in df.columns]
    sub = df[af].dropna()
    if len(sub) >= len(af)+2:
        vif_rows.append(compute_vif(sub, af, name))
vif_all = pd.concat(vif_rows, ignore_index=True) if vif_rows else pd.DataFrame()
print(vif_all.to_string(index=False))
vif_all.to_csv(os.path.join(WORK_DIR,'vif_results.csv'), index=False, encoding='utf-8-sig')


In [ ]:
# ============================================================
# STEP 2.2: Panel regression (PanelOLS Entity+Time FE) + First-Difference
# ============================================================
from linearmodels.panel import PanelOLS, FirstDifferenceOLS

panel_feats = ['Z_score','LEVERAGE','SIZE','ROA','GS']
df_panel = df[['firm_id','year']+[c for c in panel_feats if c in df.columns]].dropna()
print('Panel:', df_panel.shape, 'firms=', df_panel['firm_id'].nunique())
df_p = df_panel.set_index(['firm_id','year'])

panel_log = StringIO()

print('\n[FE] PanelOLS Entity + Time Fixed Effects (Clustered SE)')
fe = PanelOLS.from_formula(
    'Z_score ~ LEVERAGE + SIZE + ROA + GS + EntityEffects + TimeEffects',
    data=df_p
).fit(cov_type='clustered', cluster_entity=True)
print(fe.summary)
panel_log.write('=== PanelOLS (Entity+Time FE) ===\n'+str(fe.summary)+'\n\n')

print('\n[FD] First-Difference OLS')
fd = FirstDifferenceOLS.from_formula(
    'Z_score ~ LEVERAGE + SIZE + ROA + GS', data=df_p
).fit(cov_type='clustered', cluster_entity=True)
print(fd.summary)
panel_log.write('=== FirstDifferenceOLS ===\n'+str(fd.summary)+'\n\n')

with open(os.path.join(WORK_DIR,'panel_regression_results.txt'),'w',encoding='utf-8-sig') as f:
    f.write(panel_log.getvalue())
print('Saved panel_regression_results.txt')


In [ ]:
# ============================================================
# STEP 2.3: SHAP for Random Forest + Gradient Boosting
# ============================================================
import shap

SHAP_FEAT = ml_features  # use whatever the ML step settled on
shap_df = df[SHAP_FEAT+['financial_distress']].dropna()
print('SHAP dataset:', shap_df.shape)

X = shap_df[SHAP_FEAT].values; y = shap_df['financial_distress'].values
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

def shap_for(model, label, prefix):
    expl = shap.TreeExplainer(model)
    sv = expl.shap_values(Xte)
    if isinstance(sv,list):                          sv = np.array(sv[1])
    elif isinstance(sv,np.ndarray) and sv.ndim==3:    sv = sv[:,:,1]
    sv = np.array(sv)
    rank = pd.DataFrame({'Feature':SHAP_FEAT,'Mean_Abs_SHAP':np.abs(sv).mean(0)}) \
            .sort_values('Mean_Abs_SHAP', ascending=False)
    print(f'\n{label}:'); print(rank.to_string(index=False))
    plt.figure(figsize=(8,5))
    shap.summary_plot(sv, Xte, feature_names=SHAP_FEAT, plot_type='bar', show=False)
    plt.title(f'SHAP Importance - {label}'); plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR,f'shap_summary_{prefix}.png'), dpi=150, bbox_inches='tight'); plt.close()
    plt.figure(figsize=(8,5))
    shap.summary_plot(sv, Xte, feature_names=SHAP_FEAT, show=False)
    plt.title(f'SHAP Beeswarm - {label}'); plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR,f'shap_beeswarm_{prefix}.png'), dpi=150, bbox_inches='tight'); plt.close()
    return rank

rf = RandomForestClassifier(n_estimators=200,max_depth=6,random_state=42,n_jobs=-1).fit(Xtr,ytr)
shap_rf = shap_for(rf,'Random Forest','rf')
gb = GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,random_state=42).fit(Xtr,ytr)
shap_gb = shap_for(gb,'Gradient Boosting','gb')


In [ ]:
# ============================================================
# STEP 2.4: Interaction LEVERAGE x GS
# ============================================================
int_cols = [c for c in ['Z_score','financial_distress','LEVERAGE','SIZE','ROA','GS','year'] if c in df.columns]
df_int = df[int_cols].dropna().copy()
df_int['year'] = df_int['year'].astype(str)

ols_forms = {
    'OLS_M1_Baseline':    'Z_score ~ LEVERAGE + SIZE + ROA + C(year)',
    'OLS_M2_ESG':         'Z_score ~ LEVERAGE + SIZE + ROA + GS + C(year)',
    'OLS_M3_Interaction': 'Z_score ~ LEVERAGE + SIZE + ROA + GS + LEVERAGE:GS + C(year)',
}
logit_forms = {
    'Logit_M1_Baseline':    'financial_distress ~ LEVERAGE + SIZE + ROA + C(year)',
    'Logit_M2_ESG':         'financial_distress ~ LEVERAGE + SIZE + ROA + GS + C(year)',
    'Logit_M3_Interaction': 'financial_distress ~ LEVERAGE + SIZE + ROA + GS + LEVERAGE:GS + C(year)',
}

def grab(res, model_name):
    out = []
    try:
        tab = res.summary2().tables[1]
        for idx,row in tab.iterrows():
            try:
                pkey = 'P>|t|' if 'P>|t|' in row.index else 'P>|z|'
                p = float(row[pkey])
                sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
                out.append({'Model':model_name,'Variable':idx,
                            'Coef':round(float(row['Coef.']),6),
                            'StdErr':round(float(row['Std.Err.']),6),
                            'p-value':round(p,6),'Significant':sig})
            except: pass
    except: pass
    return out

rows = []
for n,f_ in ols_forms.items():
    try:
        m = smf.ols(f_, data=df_int).fit(cov_type='HC1'); rows += grab(m,n)
        if 'M3' in n:
            v = m.params.get('LEVERAGE:GS', None)
            p = m.pvalues.get('LEVERAGE:GS', None)
            if v is not None:
                print(f'\nLEVERAGE:GS in OLS = {v:+.4f} (p={p:.4f})')
    except Exception as e: print('OLS error',n,e)
for n,f_ in logit_forms.items():
    try: rows += grab(smf.logit(f_, data=df_int).fit(disp=0,maxiter=500), n)
    except Exception as e: print('Logit error',n,e)

int_df = pd.DataFrame(rows)
int_df.to_csv(os.path.join(WORK_DIR,'interaction_regression_results.csv'), index=False, encoding='utf-8-sig')
print('Saved interaction_regression_results.csv  (', len(int_df), 'rows)')


In [ ]:
# ============================================================
# STEP 2.5: Merton Distance-to-Default
# ============================================================
m_cols = ['firm_id','year','Total Assets','Total Liabilities','ROA','SIZE',
          'Z_score','distress_zone','LEVERAGE','GS']
df_m = df[[c for c in m_cols if c in df.columns]].copy()
roa_stats = df_m.groupby('firm_id')['ROA'].agg(mu_roa='mean', sigma_roa='std').reset_index()
df_m = df_m.merge(roa_stats, on='firm_id', how='left')

def dd(row):
    V,D,mu,sig = row.get('Total Assets'),row.get('Total Liabilities'),row.get('mu_roa'),row.get('sigma_roa')
    if any(pd.isna(x) for x in [V,D,mu,sig]): return np.nan
    if D<=0 or V<=0 or sig<=0: return np.nan
    return (np.log(V/D) + (mu-0.5*sig**2))/sig

df_m['merton_DD'] = df_m.apply(dd, axis=1)
df_m['merton_distress_zone'] = df_m['merton_DD'].apply(
    lambda x: 'Unknown' if pd.isna(x) else ('High_Risk' if x<0 else ('Moderate_Risk' if x<=1 else 'Low_Risk')))
print(df_m['merton_DD'].describe()); print(df_m['merton_distress_zone'].value_counts())

if 'distress_zone' in df_m.columns:
    cmp = df_m[['distress_zone','merton_distress_zone']].dropna()
    a = cmp['distress_zone'].apply(lambda x: 1 if 'Distress' in str(x) else 0)
    mm = cmp['merton_distress_zone'].apply(lambda x: 1 if x=='High_Risk' else 0)
    agreement = (a==mm).mean()
    print(f'Agreement Altman vs Merton: {agreement:.4f}')
cdf = df_m[['Z_score','merton_DD']].dropna()
if len(cdf)>5: print(f'Pearson(Z,DD) = {cdf.corr().iloc[0,1]:.4f}')

df_m[['firm_id','year','merton_DD','merton_distress_zone']].to_csv(
    os.path.join(WORK_DIR,'merton_dd_results.csv'), index=False, encoding='utf-8-sig')
df_v2 = df.merge(df_m[['firm_id','year','merton_DD','merton_distress_zone']],
                 on=['firm_id','year'], how='left')
df_v2.to_csv(os.path.join(WORK_DIR,'merged_esg_distress_panel_v2.csv'),
             index=False, encoding='utf-8-sig')
print('Saved merged_esg_distress_panel_v2.csv', df_v2.shape)


## 3. Hausman Specification Test (rank-truncated)

Reads `merged_esg_distress_panel_v2.csv`. Outputs `hausman_test_results.txt`.

In [ ]:
from linearmodels.panel import PanelOLS, RandomEffects

df_h = pd.read_csv(os.path.join(WORK_DIR,'merged_esg_distress_panel_v2.csv'), encoding='utf-8-sig')
DV  = 'Z_score'
IVs = ['LEVERAGE','SIZE','ROA','GS']
df_h = df_h[['firm_id','year',DV]+IVs].dropna().set_index(['firm_id','year'])

formula = f"{DV} ~ {' + '.join(IVs)}"

fe_robust = PanelOLS.from_formula(f"{formula} + EntityEffects + TimeEffects", data=df_h)\
              .fit(cov_type='clustered', cluster_entity=True)
re_robust = RandomEffects.from_formula(formula, data=df_h).fit(cov_type='robust')
print('FIXED EFFECTS\n', fe_robust.summary)
print('RANDOM EFFECTS\n', re_robust.summary)

# Classical Hausman with unadjusted (homoskedastic) covariance
fe_u = PanelOLS.from_formula(f"{formula} + EntityEffects + TimeEffects", data=df_h).fit(cov_type='unadjusted')
re_u = RandomEffects.from_formula(formula, data=df_h).fit(cov_type='unadjusted')
b_diff   = fe_u.params[IVs].values - re_u.params[IVs].values
cov_diff = fe_u.cov.loc[IVs,IVs].values - re_u.cov.loc[IVs,IVs].values

eigvals, eigvecs = np.linalg.eigh(cov_diff)
mev  = max(abs(eigvals)); tol = 1e-6
mask = eigvals > tol*mev; rank = int(mask.sum())
if rank == 0:
    H,p = 0.0, 1.0
else:
    Vp = eigvecs[:,mask]; Lp = eigvals[mask]
    bp = Vp.T @ b_diff
    H = float(np.sum(bp**2/Lp))
    p = 1.0 - spstats.chi2.cdf(H, df=rank)

decision = 'FE preferred' if p<0.05 else 'RE may be considered (FE retained)'
print(f'\nHausman \u03c7\u00b2(rank={rank}) = {H:.4f}, p = {p:.6f} -> {decision}')

with open(os.path.join(WORK_DIR,'hausman_test_results.txt'),'w',encoding='utf-8') as f:
    f.write(f"Hausman \u03c7\u00b2(rank={rank}) = {H:.4f}\np = {p:.6f}\nDecision: {decision}\n")
print('Saved hausman_test_results.txt')

# Cache results for the Word-update step
HAUSMAN_RESULTS = dict(rank=rank, chi2=H, pvalue=p, decision=decision, fe=fe_robust, re=re_robust)


## 4. Outputs summary

After running, the working directory should contain:

* `merged_esg_distress_panel.csv` — base panel + Z″-score
* `merged_esg_distress_panel_v2.csv` — adds Merton DD
* `correlation_heatmap.png`, `feature_importance.png`
* `vif_results.csv`
* `panel_regression_results.txt` — PanelOLS + FD-OLS
* `shap_summary_*.png`, `shap_beeswarm_*.png`
* `interaction_regression_results.csv`
* `merton_dd_results.csv`
* `hausman_test_results.txt`

Use the Files panel on the left of Colab to download whatever you need, or set
`USE_DRIVE = True` in Step 0 to write directly to Google Drive.